## Reading 32-1 - Software Correctness: Why Flight Software Fails

By the end of this reading, you will learn:
<ol>
    <li>What it means for software to be <i>correct</i>, and the difference between verification and validation</li>
    <li>Six well-documented failures in safety-critical systems, and what actually caused each one</li>
    <li>How the data representation topics from the start of the semester show up in real mission failures</li>
    <li>Why embedded and spaceflight software is held to a stricter standard than the code on your laptop</li>
</ol>

> The material in this unit is adapted from <b>Software Correctness: Writing Software for Functional Safety and Safety-Critical Systems</b> by Amalaye Oyake, a flight software engineer who has worked on Mars rovers, International Space Station payloads, NASA's first deep-space internet test, and the Blue Moon MK2 lunar lander.

### Why We Are Studying This

For most of the semester, a program was "correct" if it compiled and printed the right answer on the test cases. For the rest of the semester, we are going to raise that bar to the standard used for spacecraft.

Software has played a role in nearly every spaceflight mishap since the mid-1990s. In each incident, the pattern is clear:
<ul>
    <li>Hardware can be tested, inspected, and qualified before launch</li>
    <li>Software can pass every test you wrote and still fail on the one input you did not think of</li>
</ul>

### Getting the Reading Code

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your Reading Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir reading32
    cd reading32

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/reading/lec32/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    lec31.c   library.txt   Makefile   setup.sh   

### What Does "Correct" Actually Mean?

Correct software meets a correct specification, on time, even when something breaks. That one sentence contains three separate ideas:

<ul>
    <li><b>Verification</b> - <i>Did we build it right?</i> The code does what the specification says. We check this with code review, compiler warnings, analysis tools, and testing.</li>
    <li><b>Validation</b> - <i>Did we build the right thing?</i> The specification itself matches what the mission needs. Code can pass every test and still be wrong if the specification is wrong.</li>
    <li><b>Correct in context</b> - The answer arrives on time, survives faults, and is accurate enough for the job.</li>
</ul>

Keep the distinction between verification and validation in mind for the next section. Several of the most famous failures passed verification. The code did exactly what it was written to do. The specification was wrong.

### Six Failures Every Engineer Should Know

The table below summarizes six of the best-documented software-related failures in safety-critical systems.

<table>
    <tr><th>Incident</th><th>Year</th><th>What went wrong</th><th>Class of error</th></tr>
    <tr><td>Therac-25 radiation therapy machine</td><td>1985-87</td><td>A race condition in how the software handled fast operator input delivered massive radiation overdoses. Earlier models had hardware interlocks that would have stopped it; those were removed and the software was trusted instead.</td><td>Concurrency</td></tr>
    <tr><td>Patriot missile battery, Dhahran</td><td>1991</td><td>The system clock counted time in tenths of a second, and 0.1 was stored in a 24-bit fixed-point register. After about 100 hours of operation, the accumulated error was 0.34 seconds, enough to miss an incoming missile.</td><td>Time, numerics</td></tr>
    <tr><td>Ariane 5 Flight 501</td><td>1996</td><td>Code reused from Ariane 4 converted a 64-bit floating-point velocity into a 16-bit integer. Ariane 5 flew faster, the value overflowed, and both redundant guidance computers failed identically because they ran the same software.</td><td>Requirements, reuse</td></tr>
    <tr><td>Mars Pathfinder</td><td>1997</td><td>A low-priority task held a shared lock that a high-priority task needed, causing repeated watchdog resets on Mars. Engineers fixed it from the ground.</td><td>Concurrency</td></tr>
    <tr><td>Mars Climate Orbiter</td><td>1999</td><td>Ground software produced thrust data in pound-force seconds; the navigation software expected newton-seconds. The spacecraft was lost in the Martian atmosphere.</td><td>Units, interfaces</td></tr>
    <tr><td>Boeing 737 MAX MCAS</td><td>2018-19</td><td>An automatic trim system acted on input from a single angle-of-attack sensor, so one faulty sensor could command the aircraft.</td><td>Single point of failure</td></tr>
</table>

Notice what is <i>not</i> on this list: none of these is a classic segmentation fault. Memory bugs matter enormously. Microsoft and the Chromium project have each reported that roughly 70% of their serious security vulnerabilities are memory-safety bugs. But the failures that destroyed missions came from logic, timing, units, and assumptions.

### You Already Know Why Two of These Happened

At the beginning of the semester, we studied binary, hexadecimal, fixed point, and IEEE 754 <i>before</i> we wrote any C. Two of the failures above come straight from that material.

<b>The Patriot missile:</b> We showed that 0.1 cannot be represented exactly in binary. It is a repeating fraction, just like 1/3 in decimal. Storing it in 24 bits means chopping off the repeating part. The error in each tick is tiny, but the clock adds it ten times per second, every second, for days:
<ul>
    <li>A small error <i>times</i> a large number of operations is a large error</li>
    <li>This is the difference between <b>precision</b> (how many bits you keep) and <b>accuracy</b> (how close you are to the true value)</li>
</ul>

<b>Ariane 5:</b> A 16-bit signed integer can hold values from -32,768 to 32,767. The horizontal velocity value on Ariane 5 was larger than that. The conversion overflowed, the guidance software raised an error, and the rocket broke apart about 40 seconds after launch.

Ariane 5's software was written in Ada, a language with much stronger type checking than C. The language did exactly what it was told. No language can protect you from a requirement that says "this value will always fit in 16 bits" when it will not.

### What Makes Embedded and Flight Software Different

Your laptop forgives a lot. If a program crashes, you run it again. Flight software runs under very different rules:
<ul>
    <li><b>Hard deadlines:</b> A late answer is a wrong answer. A thruster command that arrives 50 ms late can be worse than no command at all.</li>
    <li><b>Fixed memory, power, and thermal budgets:</b> There is no "just add more RAM."</li>
    <li><b>Radiation:</b> Charged particles can flip bits in memory while the program is running.</li>
    <li><b>No operator in the loop:</b> A Mars rover is minutes of light-time away. The software must detect problems and put itself in a safe state on its own.</li>
    <li><b>Years without a reboot:</b> A counter that overflows after 49 days will eventually overflow.</li>
</ul>

The C programming language gives you direct control over memory and timing, which is exactly why it is used for this kind of software. It also gives you every opportunity to make a mistake that no one catches until it matters. Over the next eight lectures, we will learn the rules and tools that engineers use to write C that can be trusted with a mission.

### <font color = "red">Class Introduction Question #1 - What is the difference between verification and validation? Which one failed for the Mars Climate Orbiter?</font>

### <font color = "red">Class Introduction Question #2 - Using what you learned about binary fractions, explain why the Patriot missile's clock error grew the longer the system ran.</font>

### <font color = "red">Class Introduction Question #3 - The Ariane 5 had two redundant guidance computers. Why did redundancy not save the mission?</font>

### The next reading for this lecture is <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec32/Reading%2032-2.ipynb">Reading 32-2 - Undefined Behavior and the Compiler as Your First Analyzer</a>